In [0]:
-- Rebuild Silver table before reload
DROP TABLE IF EXISTS nestle_dev_silver.core.sales_transactions_scd2;

In [0]:
CREATE TABLE IF NOT EXISTS nestle_dev_silver.core.sales_transactions_scd2 (
  transaction_id STRING,
  product_id STRING,
  region STRING,
  channel STRING,
  customer_id STRING,
  quantity LONG,
  unit_price DECIMAL(10, 2),
  amount DECIMAL(15, 2),
  dbt_valid_from DATE,
  dbt_valid_to DATE,
  dbt_is_current BOOLEAN
) USING DELTA
TBLPROPERTIES (
  'comment' = 'Silver layer SCD2 for SQL Server sales transactions'
);

SELECT COUNT(*) FROM nestle_dev_silver.core.sales_transactions_scd2;

In [0]:
-- Step 1: Expire rows that have changed in Bronze
UPDATE nestle_dev_silver.core.sales_transactions_scd2 AS target
SET dbt_valid_to = current_date(),
    dbt_is_current = FALSE
WHERE dbt_is_current = TRUE
  AND EXISTS (
    SELECT 1 FROM nestle_dev_bronze.sql_server.sales_transactions AS src
    WHERE src.transaction_id = target.transaction_id
      AND (src.amount <> target.amount 
           OR src.quantity <> target.quantity
           OR src.channel <> target.channel)
  );

SELECT COUNT(*) as expired_rows FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = FALSE;

In [0]:
-- Step 2: Insert latest current rows from deduplicated Bronze
INSERT INTO nestle_dev_silver.core.sales_transactions_scd2
WITH bronze_deduped AS (
  SELECT
    transaction_id,
    product_id,
    region,
    channel,
    customer_id,
    quantity,
    CAST(unit_price AS DECIMAL(10, 2)) AS unit_price,
    CAST(amount AS DECIMAL(15, 2)) AS amount,
    ROW_NUMBER() OVER (
      PARTITION BY transaction_id
      ORDER BY ingestion_timestamp DESC,
               TO_TIMESTAMP(modified_at) DESC,
               TO_TIMESTAMP(created_at) DESC
    ) AS rn
  FROM nestle_dev_bronze.sql_server.sales_transactions
)
SELECT
  src.transaction_id,
  src.product_id,
  src.region,
  src.channel,
  src.customer_id,
  src.quantity,
  src.unit_price,
  src.amount,
  CAST(TO_TIMESTAMP(src.modified_at) AS DATE) AS dbt_valid_from,
  CAST('2099-12-31' AS DATE) AS dbt_valid_to,
  TRUE AS dbt_is_current
FROM bronze_deduped src
WHERE src.rn = 1
  AND NOT EXISTS (
    SELECT 1 FROM nestle_dev_silver.core.sales_transactions_scd2 tgt
    WHERE tgt.transaction_id = src.transaction_id
      AND tgt.dbt_is_current = TRUE
  );

SELECT COUNT(*) as inserted_rows FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = TRUE;

In [0]:
SELECT 
  COUNT(*) as total_rows,
  COUNT(CASE WHEN dbt_is_current = TRUE THEN 1 END) as current_rows,
  COUNT(CASE WHEN dbt_is_current = FALSE THEN 1 END) as expired_rows,
  MIN(dbt_valid_from) as earliest_valid_from,
  MAX(dbt_valid_to) as latest_valid_to
FROM nestle_dev_silver.core.sales_transactions_scd2;